# Foundations A4: Autograd and the Training Loop

Companion notebook for [Foundations A4 of the interactive course](https://bc-sparknet.github.io/make-an-llm-2/#/chapter/autograd). The code cells follow the chapter's examples in order; the explanations are on the site.

*This notebook is generated from the chapter source. To change it, edit the chapter and run `npm run notebooks`.*

In [ ]:
%pip install -q torch tiktoken

## What training means

## Gradients: which way is downhill

In [ ]:
def loss_fn(w):
    return (w - 3) ** 2

def grad_fn(w):
    return 2 * (w - 3)          # the slope of the loss at w

w = 0.0
lr = 0.1
for step in range(5):
    w = w - lr * grad_fn(w)     # one gradient-descent step
    print(f"step {step}: w={w:.3f} loss={loss_fn(w):.3f}")
# step 0: w=0.600 loss=5.760
# step 1: w=1.080 loss=3.686
# step 2: w=1.464 loss=2.359
# step 3: w=1.771 loss=1.510
# step 4: w=2.017 loss=0.966

In [ ]:
for lr in [0.01, 0.1, 1.1]:
    w = 0.0
    for step in range(20):
        w = w - lr * grad_fn(w)
    print(f"lr={lr}: w after 20 steps = {w:.2f}")
# lr=0.01: w after 20 steps = 1.00      too slow: still far from 3
# lr=0.1: w after 20 steps = 2.97       close to the answer
# lr=1.1: w after 20 steps = -112.01    diverged

## Autograd: gradients for free

In [ ]:
import torch

x = torch.tensor(2.0)
y = torch.tensor(4.0)
w = torch.tensor(1.0, requires_grad=True)    # parameters: track gradients
b = torch.tensor(0.5, requires_grad=True)

pred = w * x + b
loss = (pred - y) ** 2
print(pred)   # tensor(2.5000, grad_fn=<AddBackward0>)
print(loss)   # tensor(2.2500, grad_fn=<PowBackward0>)

In [ ]:
loss.backward()
print(w.grad)   # tensor(-6.)
print(b.grad)   # tensor(-3.)

### Gradients accumulate

In [ ]:
loss = (w * x + b - y) ** 2   # rebuild the graph (backward() frees the old one)
loss.backward()
print(w.grad)   # tensor(-12.)  = -6 + -6: added to the old value!

w.grad.zero_()                # reset to zero in place
b.grad.zero_()
loss = (w * x + b - y) ** 2
loss.backward()
print(w.grad)   # tensor(-6.)   correct again

### Fitting a line, by hand

In [ ]:
xs = torch.tensor([0.0, 1.0, 2.0, 3.0, 4.0, 5.0])
ys = torch.tensor([1.2, 2.8, 5.1, 7.3, 8.7, 11.2])

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.02

for step in range(301):
    pred = w * xs + b
    loss = ((pred - ys) ** 2).mean()     # mean squared error
    loss.backward()                      # fills w.grad and b.grad

    with torch.no_grad():                # the update itself shouldn't be tracked
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_()
    b.grad.zero_()

    if step % 100 == 0:
        print(f"step {step:3d}: loss={loss.item():.4f} w={w.item():.3f} b={b.item():.3f}")
# step   0: loss=48.2850 w=0.838 b=0.242
# step 100: loss=0.0553 w=2.036 b=0.919
# step 200: loss=0.0497 w=2.009 b=1.015
# step 300: loss=0.0492 w=2.001 b=1.044

### Switching tracking off: `no_grad()` and `detach()`

In [ ]:
with torch.no_grad():
    guess = w * 6.0 + b
print(guess.requires_grad)   # False: no graph was built

tracked = w * 6.0 + b
print(tracked.requires_grad)            # True
print(tracked.detach().requires_grad)   # False
print(f"{guess.item():.2f}")            # 13.05: the fitted line's prediction at x = 6

## Models as `nn.Module`

In [ ]:
import torch.nn as nn

class TinyMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()                          # always call this first
        self.layer1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.layer2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x):
        h = self.drop(self.act(self.layer1(x)))
        return self.layer2(h)

torch.manual_seed(123)
model = TinyMLP(in_dim=2, hidden_dim=16, out_dim=2)

batch = torch.randn(5, 2)       # 5 examples, 2 features each
logits = model(batch)           # calling the model runs forward()
print(logits.shape)             # torch.Size([5, 2])

In [ ]:
for name, p in model.named_parameters():
    print(name, tuple(p.shape))
# layer1.weight (16, 2)
# layer1.bias (16,)
# layer2.weight (2, 16)
# layer2.bias (2,)

print(sum(p.numel() for p in model.parameters()))   # 82

### `model.train()` and `model.eval()`

In [ ]:
drop = nn.Dropout(p=0.5)
ones = torch.ones(8)

torch.manual_seed(0)
drop.train()
print(drop(ones))   # tensor([0., 0., 2., 0., 0., 0., 2., 2.]): random zeros, survivors ×2
drop.eval()
print(drop(ones))   # tensor([1., 1., 1., 1., 1., 1., 1., 1.]): does nothing

## Optimizers

In [ ]:
torch.manual_seed(123)
X = xs.unsqueeze(1)             # shape (6, 1): 6 examples, 1 feature
Y = ys.unsqueeze(1)

line = nn.Linear(1, 1)          # holds w (weight) and b (bias)
optimizer = torch.optim.SGD(line.parameters(), lr=0.02)
loss_fn = nn.MSELoss()

for step in range(301):
    optimizer.zero_grad()       # 1. clear old gradients
    pred = line(X)              # 2. forward
    loss = loss_fn(pred, Y)     # 3. loss
    loss.backward()             # 4. backward: compute gradients
    optimizer.step()            # 5. update the parameters
    if step % 100 == 0:
        print(f"step {step:3d}: loss={loss.item():.4f}")
# step   0: loss=66.4278
# step 100: loss=0.0522
# step 200: loss=0.0494
# step 300: loss=0.0492

print(f"w={line.weight.item():.3f} b={line.bias.item():.3f}")   # w=2.000 b=1.048

## Feeding data: `Dataset` and `DataLoader`

In [ ]:
from torch.utils.data import Dataset, DataLoader

class PointsDataset(Dataset):
    def __init__(self, points, labels):
        self.points = points
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.points[i], self.labels[i]

torch.manual_seed(0)
toy = PointsDataset(torch.randn(10, 2), torch.arange(10) % 2)
print(len(toy))   # 10
print(toy[3])     # (tensor([-0.3414,  1.8530]), tensor(1))

loader = DataLoader(toy, batch_size=4, shuffle=True, drop_last=True)
print(len(loader))   # 2: two full batches of 4; the leftover 2 are dropped

for epoch in range(2):
    for xb, yb in loader:
        print(f"epoch {epoch}: batch {tuple(xb.shape)} labels {yb.tolist()}")
# epoch 0: batch (4, 2) labels [1, 1, 0, 0]
# epoch 0: batch (4, 2) labels [0, 1, 1, 0]
# epoch 1: batch (4, 2) labels [0, 1, 1, 1]
# epoch 1: batch (4, 2) labels [0, 0, 0, 0]

## A complete training run

In [ ]:
def make_circle_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    points = torch.rand(n, 2, generator=g) * 4 - 2     # uniform in the square [-2, 2]
    labels = (points.norm(dim=1) < 1.2).long()        # 1 = inside the circle
    return points, labels

train_x, train_y = make_circle_data(400, seed=1)
val_x, val_y = make_circle_data(200, seed=2)
print(train_x.shape, f"{train_y.float().mean():.2f}")   # torch.Size([400, 2]) 0.32

train_loader = DataLoader(PointsDataset(train_x, train_y), batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(PointsDataset(val_x, val_y), batch_size=100)

In [ ]:
def accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            preds = model(xb).argmax(dim=1)    # the class with the highest score
            correct += (preds == yb).sum().item()
            total += len(yb)
    model.train()
    return correct / total

In [ ]:
torch.manual_seed(123)
model = TinyMLP(in_dim=2, hidden_dim=16, out_dim=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()

print(f"before training: val acc {accuracy(model, val_loader):.2f}")   # 0.45

for epoch in range(20):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 4 == 0:
        avg = total_loss / len(train_loader)
        print(f"epoch {epoch+1:2d}: loss {avg:.3f}, "
              f"train acc {accuracy(model, train_loader):.2f}, val acc {accuracy(model, val_loader):.2f}")
# epoch  4: loss 0.456, train acc 0.71, val acc 0.78
# epoch  8: loss 0.261, train acc 0.93, val acc 0.94
# epoch 12: loss 0.198, train acc 0.97, val acc 0.96
# epoch 16: loss 0.172, train acc 0.99, val acc 0.97
# epoch 20: loss 0.167, train acc 0.98, val acc 0.98

### Saving and loading

In [ ]:
print(list(model.state_dict().keys()))
# ['layer1.weight', 'layer1.bias', 'layer2.weight', 'layer2.bias']

torch.save(model.state_dict(), "tiny_mlp.pth")

restored = TinyMLP(in_dim=2, hidden_dim=16, out_dim=2)   # same architecture, random weights
restored.load_state_dict(torch.load("tiny_mlp.pth"))     # now the trained weights
restored.eval()

with torch.no_grad():
    print(restored(torch.tensor([[0.0, 0.0], [1.8, 1.8]])).argmax(dim=1))
# tensor([1, 0]): the centre is inside, the corner is outside
print(accuracy(restored, val_loader))   # 0.985